In [2]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

DATA_PATH = Path("final/ml_data_final")

# Проверяем путь
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Путь не найден: {DATA_PATH.resolve()}")

# Находим все Parquet-файлы
files = sorted(DATA_PATH.rglob("*.parquet"))

print(f"Найдено Parquet-файлов: {len(files)}")

if not files:
    raise ValueError("В папке нет Parquet-файлов")

# Считаем строки без загрузки всего датасета
total_rows = sum(
    pq.read_metadata(file).num_rows
    for file in files
)

print(f"Всего строк: {total_rows:,}")

# Читаем первый файл для предварительного анализа
df = pd.read_parquet(files[0])

print("\nРазмер первого файла:", df.shape)

print("\nНазвания столбцов:")
print(df.columns.tolist())

print("\nТипы данных:")
print(df.dtypes.to_string())

print("\nПропуски в первом файле:")
print(
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .head(30)
)

print("\nПервые 5 строк:")
display(df.head())

print("\nЧисло дубликатов в первом файле:")
print(df.duplicated().sum())

Найдено Parquet-файлов: 1600
Всего строк: 13,327,942

Размер первого файла: (2799, 27)

Названия столбцов:
['borrower_id', 'episode_id', 'origination_month_index', 'month_on_book', 'age_months', 'industry', 'entity_type', 'monthly_revenue', 'revenue_trend_12m', 'revenue_volatility', 'operating_margin', 'cash_buffer_months', 'dscr', 'debt_to_revenue', 'prior_overdue_30d', 'requested_amount', 'term_months', 'annual_rate', 'macro_stress', 'default_event', 'current_monthly_revenue', 'current_operating_margin', 'current_cash_buffer_months', 'current_dscr', 'current_debt_to_revenue', 'current_revenue_trend_3m', 'current_macro_stress']

Типы данных:
borrower_id                     int64
episode_id                      int64
origination_month_index         int64
month_on_book                   int64
age_months                      int64
industry                          str
entity_type                       str
monthly_revenue               float64
revenue_trend_12m             float64
revenue

,borrower_id,episode_id,origination_month_index,month_on_book,age_months,industry,entity_type,monthly_revenue,revenue_trend_12m,revenue_volatility,...,annual_rate,macro_stress,default_event,current_monthly_revenue,current_operating_margin,current_cash_buffer_months,current_dscr,current_debt_to_revenue,current_revenue_trend_3m,current_macro_stress
0,6,6,72,1,147,transport,OOO,2.262810e+06,0.045550,0.204102,...,0.148629,0.090746,0,2.262810e+06,0.218372,7.790176,6.175806,0.228488,-0.012237,0.090746
1,12,12,81,1,124,transport,IP,4.917488e+05,-0.094909,0.134201,...,0.171312,1.108760,0,4.917488e+05,0.233010,7.065001,8.000000,0.162282,0.130395,1.108760
2,24,24,77,1,134,manufacturing,OOO,2.805299e+06,-0.312378,0.136244,...,0.163799,1.029945,0,2.805299e+06,0.050487,4.510523,1.533400,0.149619,-0.237714,1.029945
3,35,35,74,1,97,retail,IP,5.000249e+06,-0.005845,0.074359,...,0.194287,0.950372,0,5.000249e+06,0.205946,8.817418,4.089648,0.311267,0.037114,0.950372
4,37,37,72,1,86,manufacturing,OOO,1.539147e+06,-0.029966,0.086863,...,0.204019,0.090746,0,1.539147e+06,0.162989,5.318985,3.325208,0.329624,0.074791,0.090746



Число дубликатов в первом файле:
0


In [3]:
from pathlib import Path
import duckdb

DATA_PATH = Path("final/ml_data_final").resolve()

parquet_pattern = (
    DATA_PATH.as_posix().rstrip("/") + "/**/*.parquet"
)

con = duckdb.connect()

con.execute("SET threads TO 4")

query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT borrower_id) AS unique_borrowers,
    COUNT(DISTINCT (borrower_id, episode_id)) AS unique_episodes,

    SUM(CASE WHEN default_event = 1 THEN 1 ELSE 0 END)
        AS default_rows,

    MIN(month_on_book) AS min_month_on_book,
    MAX(month_on_book) AS max_month_on_book,

    MIN(age_months) AS min_age_months,
    MAX(age_months) AS max_age_months

FROM read_parquet(?, union_by_name = true)
"""

result = con.execute(
    query, [parquet_pattern]
).df()

display(result)

,total_rows,unique_borrowers,unique_episodes,default_rows,min_month_on_book,max_month_on_book,min_age_months,max_age_months
0,13327942,200000,400000,114558.0,1,60,3,537


In [4]:
query = """
WITH episodes AS (
    SELECT
        borrower_id,
        episode_id,

        COUNT(*) AS observation_count,

        SUM(
            CASE WHEN default_event = 1
            THEN 1 ELSE 0 END
        ) AS default_count,

        MIN(month_on_book) AS first_month,
        MAX(month_on_book) AS last_month

    FROM read_parquet(?, union_by_name = true)

    GROUP BY borrower_id, episode_id
)

SELECT
    COUNT(*) AS total_episodes,

    COUNT(*) FILTER (
        WHERE default_count = 0
    ) AS episodes_without_default,

    COUNT(*) FILTER (
        WHERE default_count = 1
    ) AS episodes_with_one_default,

    COUNT(*) FILTER (
        WHERE default_count > 1
    ) AS episodes_with_multiple_defaults,

    COUNT(*) FILTER (
        WHERE observation_count > 1
    ) AS episodes_with_multiple_observations,

    MIN(first_month) AS min_first_month,
    MAX(last_month) AS max_last_month

FROM episodes
"""

display(
    con.execute(query, [parquet_pattern]).df()
)

,total_episodes,episodes_without_default,episodes_with_one_default,episodes_with_multiple_defaults,episodes_with_multiple_observations,min_first_month,max_last_month
0,400000,342721,0,57279,400000,1,60


In [5]:
query = """
WITH episodes AS (
    SELECT
        borrower_id,
        episode_id,

        MIN(month_on_book) AS first_month,
        MAX(month_on_book) AS last_month,

        MIN(month_on_book)
            FILTER (WHERE default_event = 1)
            AS first_default_month,

        MAX(month_on_book)
            FILTER (WHERE default_event = 1)
            AS last_default_month,

        COUNT(*) AS observation_count,
        COUNT(*) FILTER (WHERE default_event = 1)
            AS default_rows

    FROM read_parquet(?, union_by_name = true)

    GROUP BY borrower_id, episode_id
)

SELECT
    COUNT(*) AS total_episodes,

    COUNT(*) FILTER (
        WHERE first_month <> 1
    ) AS episodes_not_starting_at_one,

    COUNT(*) FILTER (
        WHERE first_default_month IS NOT NULL
    ) AS default_episodes,

    COUNT(*) FILTER (
        WHERE first_default_month IS NOT NULL
          AND last_default_month = last_month
    ) AS default_continues_to_end,

    MIN(first_default_month) AS earliest_default_month,
    MAX(first_default_month) AS latest_default_month,

    ROUND(AVG(observation_count), 2)
        AS avg_observations_per_episode,

    ROUND(AVG(default_rows), 2)
        AS avg_default_rows_per_episode

FROM episodes
"""

display(con.execute(query, [parquet_pattern]).df())

,total_episodes,episodes_not_starting_at_one,default_episodes,default_continues_to_end,earliest_default_month,latest_default_month,avg_observations_per_episode,avg_default_rows_per_episode
0,400000,0,57279,57279,1,60,33.32,0.29


In [6]:
query = """
WITH first_defaults AS (
    SELECT
        borrower_id,
        episode_id,
        MIN(month_on_book) AS default_month

    FROM read_parquet(?, union_by_name = true)

    WHERE default_event = 1

    GROUP BY borrower_id, episode_id
)

SELECT
    CASE
        WHEN default_month <= 12 THEN '01–12'
        WHEN default_month <= 24 THEN '13–24'
        WHEN default_month <= 36 THEN '25–36'
        WHEN default_month <= 48 THEN '37–48'
        ELSE '49–60'
    END AS default_period,

    COUNT(*) AS default_episodes

FROM first_defaults

GROUP BY 1
ORDER BY 1
"""

display(con.execute(query, [parquet_pattern]).df())

,default_period,default_episodes
0,01–12,30665
1,13–24,16496
2,25–36,7041
3,37–48,2425
4,49–60,652


In [7]:
query = """
WITH episode_stats AS (
    SELECT
        borrower_id,
        episode_id,

        MIN(month_on_book) AS first_month,
        MAX(month_on_book) AS last_month,

        MIN(month_on_book)
            FILTER (WHERE default_event = 1)
            AS first_default_month,

        COUNT(DISTINCT monthly_revenue)
            AS revenue_values,

        COUNT(DISTINCT dscr)
            AS dscr_values,

        COUNT(DISTINCT requested_amount)
            AS amount_values,

        MAX(term_months) AS term_months

    FROM read_parquet(?, union_by_name = true)

    GROUP BY borrower_id, episode_id
)

SELECT
    COUNT(*) AS total_episodes,

    COUNT(*) FILTER (
        WHERE first_default_month IS NOT NULL
          AND last_month > first_default_month
    ) AS observations_after_default,

    COUNT(*) FILTER (
        WHERE first_default_month IS NULL
          AND last_month < term_months
    ) AS potentially_censored_episodes,

    COUNT(*) FILTER (
        WHERE revenue_values > 1
    ) AS episodes_with_changing_revenue,

    COUNT(*) FILTER (
        WHERE dscr_values > 1
    ) AS episodes_with_changing_dscr,

    COUNT(*) FILTER (
        WHERE amount_values > 1
    ) AS episodes_with_changing_amount

FROM episode_stats
"""

display(con.execute(query, [parquet_pattern]).df())

,total_episodes,observations_after_default,potentially_censored_episodes,episodes_with_changing_revenue,episodes_with_changing_dscr,episodes_with_changing_amount
0,400000,0,44292,0,0,0
